## Libraries

In [ ]:
# Prevent OpenMP runtime conflict between libraries like PyTorch, TensorFlow, and NumPy on Windows
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchmetrics.classification import MulticlassAccuracy

from tqdm import tqdm
import numpy as np

from danflow.training import Trainer

## Load Data

In [ ]:
data = torch.load('../saved_values/shuttle_data.pt', weights_only=False)

x_train = data["x_train"]
y_train = data["y_train"]

x_valid = data["x_valid"]
y_valid = data["y_valid"]

x_test = data["x_test"]
y_test = data["y_test"]

## Convert to Tensors

In [ ]:
x_train = torch.tensor(np.asarray(x_train), dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)

x_valid = torch.tensor(np.asarray(x_valid), dtype=torch.float32)
y_valid = torch.tensor(np.asarray(y_valid), dtype=torch.long)

x_test = torch.tensor(np.asarray(x_test), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

## Data Loader

In [ ]:
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=256)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

## MLP Model

The model is a 3-layer MLP with two hidden layers containing 64 and 32 neurons, respectively, using ReLU activation functions. The output layer contains 7 neurons, corresponding to the 7 target classes.

In [ ]:
def mlp_model():
    "Initializes multi layer perceptron model"
    in_features = 9
    num_class = 7
    h1 = 64
    h2 = 32

    model = nn.Sequential(nn.Linear(in_features, h1),
                           nn.ReLU(),
                           nn.Linear(h1, h2),
                           nn.ReLU(),
                           nn.Linear(h2, num_class))

    return model


model = mlp_model()
model

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=7, bias=True)
)

## Cross-Entropy Loss
$$
\mathcal{L} = -\log(\hat{y}_{\text{true}})
$$

In [ ]:
loss_fn = nn.CrossEntropyLoss()

## Optimizer

In [ ]:
optimizer = optim.SGD(model.parameters(),
                      lr=0.01,
                      momentum=0.9,
                      nesterov=True,
                      weight_decay=1e-4)

## Model Verification